# RoCE bandwidth testing from this laptop

This notebook runs `ib_write_bw` on two remote devbox nodes over your existing SSH aliases. The laptop only starts the commands and collects their output; the measured data travels directly between the two nodes over RoCE.

The measured data path is:

```text
client CPU RAM
  -> client PCIe Gen5 x16 host link
  -> client PCIe switch
  -> client PCIe Gen6 x16 NIC link
  -> client RoCE NIC
  -> network
  -> server RoCE NIC
  -> server PCIe Gen6 x16 NIC link
  -> server PCIe switch
  -> server PCIe Gen5 x16 host link
  -> server CPU RAM
```

The host-facing Gen5 x16 link has about 63 GB/s of theoretical one-way capacity. The measured 59.4 GB/s is therefore close to this test path's PCIe limit, rather than the bonded NIC's 100 GB/s network limit.

A GPU-direct NCCL transfer can use a different local path:

```text
GPU memory -> Gen6 x16 GPU link -> PCIe switch -> Gen6 x16 NIC link -> RoCE NIC
```

That path can avoid the slower host-facing Gen5 link when GPUDirect RDMA and PCIe peer-to-peer routing are active. This notebook does not use GPUs, CUDA, PyTorch, or NCCL, so a separate NCCL test is required to verify the GPU-direct path and its bandwidth.

## Units

All notebook results are shown in **GB/s**. One physical member of these bonds has a theoretical maximum of 50 GB/s, and the two-member bond has a theoretical maximum of 100 GB/s.


## Arguments you control

- `rail`: Which bonded RoCE device to test. Rail `0` means `mlx5_bond_0`; this devbox has rails 0 through 7.
- `qps`: Number of queue pairs, or independent RoCE flows, used on that rail. Start with 1, then compare with 2, 4, and 8. More QPs give the two-link bond more flows to distribute, but do not guarantee an even split.
- `message_size_mb`: Bytes submitted in each RDMA write. Use 8 MiB for a large-payload bandwidth test.
- `duration_seconds`: How long to measure after setup. Eight seconds is normally enough.

Arguments normally left alone:

- `gid_index=3`: Selects the routable RoCE-v2 address on these nodes.
- `port`: A temporary TCP control port used only to exchange test metadata. The measured payload does not use this TCP connection. A random high port is selected by default.


In [1]:
# Change these values when the devbox changes.
SERVER_HOST = "tj-w5y89m3"
CLIENT_HOST = "tj-w5y89m3-1"
SERVER_IP = "10.1.77.34"

# This was verified as the routable RoCE-v2 GID on this devbox.
GID_INDEX = 3


In [2]:
import random
import shlex
import subprocess
import time
from typing import Iterable


def check_nodes(
    server_host: str = SERVER_HOST,
    client_host: str = CLIENT_HOST,
) -> None:
    """Verify SSH and ib_write_bw before starting a benchmark."""
    for role, host in (("server", server_host), ("client", client_host)):
        result = subprocess.run(
            ["ssh", host, "hostname && command -v ib_write_bw"],
            text=True,
            capture_output=True,
            timeout=15,
        )
        if result.returncode != 0:
            raise RuntimeError(
                f"Could not prepare {role} {host}:\n{result.stdout}{result.stderr}"
            )
        print(f"{role:>6}: {host} -> {result.stdout.strip().replace(chr(10), ' | ')}")


def _ib_write_bw_command(
    *,
    device: str,
    qps: int,
    message_size_mb: float,
    duration_seconds: int,
    gid_index: int,
    port: int,
    server_ip: str | None,
) -> str:
    message_size_bytes = round(message_size_mb * 1024 * 1024)
    args = [
        "timeout",
        "--signal=TERM",
        "--kill-after=2s",
        f"{duration_seconds + 10}s",
        "ib_write_bw",
        "-d", device,             # Bonded RoCE device.
        "-F",                     # Suppress an irrelevant CPU-frequency warning.
        "-x", str(gid_index),     # Routable RoCE-v2 GID.
        "--report_gbits",         # Stable perftest output; converted to GB/s below.
        "-D", str(duration_seconds),
        "-q", str(qps),
        "-s", str(message_size_bytes),
        "-p", str(port),
    ]
    if server_ip is not None:
        args.append(server_ip)
    return shlex.join(args)


def _parse_result(output: str) -> dict:
    rows = []
    for line in output.splitlines():
        fields = line.split()
        if len(fields) != 5 or not fields[0].isdigit():
            continue
        try:
            rows.append(
                {
                    "message_size_bytes": int(fields[0]),
                    "iterations": int(fields[1]),
                    "GBps": float(fields[3]) / 8,
                    "message_rate_mpps": float(fields[4]),
                }
            )
        except ValueError:
            continue
    if not rows:
        raise RuntimeError(f"Could not find a bandwidth result in output:\n{output}")
    return rows[-1]


def _stop_process(process: subprocess.Popen) -> None:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            process.kill()


def run_ib_write_bw(
    *,
    rail: int = 0,
    qps: int = 1,
    message_size_mb: float = 8,
    duration_seconds: int = 8,
    server_host: str = SERVER_HOST,
    client_host: str = CLIENT_HOST,
    server_ip: str = SERVER_IP,
    gid_index: int = GID_INDEX,
    port: int | None = None,
    show_raw_output: bool = False,
) -> dict:
    """Run one unidirectional CPU-memory RDMA-write bandwidth test.

    Data moves from CLIENT_HOST to SERVER_HOST on mlx5_bond_<rail>.
    """
    if not 0 <= rail <= 7:
        raise ValueError("rail must be between 0 and 7")
    if qps < 1:
        raise ValueError("qps must be at least 1")
    if message_size_mb <= 0 or duration_seconds < 1:
        raise ValueError("message_size_mb and duration_seconds must be positive")
    if port is None:
        port = random.randint(30000, 50000)

    device = f"mlx5_bond_{rail}"
    server_command = _ib_write_bw_command(
        device=device, qps=qps, message_size_mb=message_size_mb,
        duration_seconds=duration_seconds, gid_index=gid_index,
        port=port, server_ip=None,
    )
    client_command = _ib_write_bw_command(
        device=device, qps=qps, message_size_mb=message_size_mb,
        duration_seconds=duration_seconds, gid_index=gid_index,
        port=port, server_ip=server_ip,
    )

    print(
        f"Testing {device}: qps={qps}, message={message_size_mb} MiB, "
        f"duration={duration_seconds}s"
    )
    server = subprocess.Popen(
        ["ssh", server_host, server_command],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    try:
        time.sleep(2)
        client = subprocess.run(
            ["ssh", client_host, client_command],
            text=True, capture_output=True,
            timeout=duration_seconds + 15,
        )
        server_output, _ = server.communicate(timeout=duration_seconds + 8)
    finally:
        _stop_process(server)

    client_output = client.stdout + client.stderr
    if client.returncode != 0 or server.returncode != 0:
        raise RuntimeError(
            f"Test failed.\nSERVER:\n{server_output}\nCLIENT:\n{client_output}"
        )
    result = _parse_result(client_output)
    result.update(
        rail=rail, qps=qps, message_size_mb=message_size_mb,
        duration_seconds=duration_seconds, server_command=server_command,
        client_command=client_command,
    )
    print(f"Result: {result['GBps']:.2f} GB/s")
    if show_raw_output:
        print(f"\nSERVER OUTPUT:\n{server_output}\nCLIENT OUTPUT:\n{client_output}")
    return result


def compare_qps(
    qps_values: Iterable[int] = (1, 2, 4, 8),
    **test_args,
) -> list[dict]:
    """Run the same rail test sequentially with different QP counts."""
    results = [run_ib_write_bw(qps=qps, **test_args) for qps in qps_values]
    print("\n QPs      GB/s")
    for result in results:
        print(f"{result['qps']:>4}  {result['GBps']:>8.2f}")
    return results


## 1. Check connectivity

Run this after creating or restarting a devbox. It does not send benchmark traffic.


In [3]:
check_nodes()

server: tj-w5y89m3 -> b300-1-5x4eyifb-0008 | /usr/bin/ib_write_bw
client: tj-w5y89m3-1 -> b300-1-s58nc356-0011 | /usr/bin/ib_write_bw


## 2. Test one rail

This sends traffic from node 1 to node 0 over `mlx5_bond_0`. Start with one QP so the result is easy to interpret.


In [4]:
one_qp_result = run_ib_write_bw(
    rail=0,
    qps=1,
    message_size_mb=8,
    duration_seconds=8,
)
one_qp_result

Testing mlx5_bond_0: qps=1, message=8 MiB, duration=8s
Result: 46.17 GB/s


{'message_size_bytes': 8388608,
 'iterations': 22016,
 'GBps': 46.17,
 'message_rate_mpps': 0.005504,
 'rail': 0,
 'qps': 1,
 'message_size_mb': 8,
 'duration_seconds': 8,
 'server_command': 'timeout --signal=TERM --kill-after=2s 18s ib_write_bw -d mlx5_bond_0 -F -x 3 --report_gbits -D 8 -q 1 -s 8388608 -p 43897',
 'client_command': 'timeout --signal=TERM --kill-after=2s 18s ib_write_bw -d mlx5_bond_0 -F -x 3 --report_gbits -D 8 -q 1 -s 8388608 -p 43897 10.1.77.34'}

## 3. Compare QP counts

These tests run one after another, not simultaneously, so they make a clean comparison. On a two-member LACP bond, multiple QPs may allow traffic to use both physical links.


In [5]:
qp_results = compare_qps(
    qps_values=(1, 2, 4, 8),
    rail=0,
    message_size_mb=8,
    duration_seconds=8,
)

Testing mlx5_bond_0: qps=1, message=8 MiB, duration=8s
Result: 47.48 GB/s
Testing mlx5_bond_0: qps=2, message=8 MiB, duration=8s
Result: 48.61 GB/s
Testing mlx5_bond_0: qps=4, message=8 MiB, duration=8s
Result: 59.41 GB/s
Testing mlx5_bond_0: qps=8, message=8 MiB, duration=8s
Result: 59.37 GB/s

 QPs      GB/s
   1     47.48
   2     48.61
   4     59.41
   8     59.37


## Interpreting results

- Around **45-50 GB/s** means the test is approximately filling one 400G physical link.
- Moving toward **90-100 GB/s** means traffic is using both 400G members of the bond efficiently.
- A higher `ib_write_bw` QP count does not change NCCL settings. The NCCL section below controls NCCL separately.
- This uses CPU memory. GPU-direct NCCL can behave differently because its source and destination buffers live in GPU memory.
- If a test fails, rerun with `show_raw_output=True` to print the complete server and client output.


# GPU-direct NCCL RoCE test

This section measures a one-way NCCL transfer from GPU memory on node 0 to GPU memory on node 1. For `rail=0`, it pins the process to GPU 0 and pins NCCL to `mlx5_bond_0`. NCCL has confirmed that this path uses `GDRDMA`, so it avoids the CPU-memory Gen5 bottleneck measured above.

The three tuning arguments map directly to:

- `qps_per_connection` -> `NCCL_IB_QPS_PER_CONNECTION`
- `split_data_on_qps` -> `NCCL_IB_SPLIT_DATA_ON_QPS`
- `channels_per_net_peer` -> `NCCL_NCHANNELS_PER_NET_PEER`

Every result is reported in **GB/s**. The function also checks NCCL's logs for `GDRDMA`; a result is rejected if NCCL does not confirm GPU-direct RDMA.


In [8]:
import json
from pathlib import Path

LOCAL_NCCL_BENCHMARK = Path(
    "/Users/jackrao/Documents/trainers/experiment_artefacts/glm/lps_1062_perf/tools/nccl_roce_p2p_bw.py"
)
REMOTE_NCCL_BENCHMARK = (
    "/root/.cache/user_artifacts/devboxes/w5y89m3/nccl_roce_p2p_bw.py"
)
REMOTE_TORCHRUN = "/root/.devbox-venvs/server/bin/torchrun"


def _nccl_command(
    *,
    node_rank: int,
    master_port: int,
    rail: int,
    payload_mb: float,
    warmup: int,
    iterations: int,
    qps_per_connection: int,
    split_data_on_qps: int,
    channels_per_net_peer: int,
) -> str:
    environment = {
        "CUDA_VISIBLE_DEVICES": str(rail),
        "NCCL_SOCKET_IFNAME": "eth0",
        "NCCL_IB_HCA": f"=mlx5_bond_{rail}:1",
        "NCCL_IB_GID_INDEX": str(GID_INDEX),
        "NCCL_IB_QPS_PER_CONNECTION": str(qps_per_connection),
        "NCCL_IB_SPLIT_DATA_ON_QPS": str(split_data_on_qps),
        "NCCL_NCHANNELS_PER_NET_PEER": str(channels_per_net_peer),
        "NCCL_DEBUG": "INFO",
        "NCCL_DEBUG_SUBSYS": "INIT,NET",
        "TORCH_NCCL_BLOCKING_WAIT": "1",
    }
    args = [
        "env",
        *(f"{name}={value}" for name, value in environment.items()),
        "timeout", "--signal=TERM", "--kill-after=5s", "120s",
        REMOTE_TORCHRUN,
        "--nnodes=2",
        "--nproc-per-node=1",
        f"--node-rank={node_rank}",
        f"--master-addr={SERVER_IP}",
        f"--master-port={master_port}",
        REMOTE_NCCL_BENCHMARK,
        f"--payload-mb={payload_mb}",
        f"--warmup={warmup}",
        f"--iterations={iterations}",
    ]
    return shlex.join(args)


def _parse_nccl_result(output: str) -> dict:
    for line in output.splitlines():
        if line.startswith("RESULT_JSON="):
            return json.loads(line.removeprefix("RESULT_JSON="))
    raise RuntimeError(f"NCCL benchmark produced no result:\n{output}")


def run_nccl_roce_bw(
    *,
    rail: int = 0,
    payload_mb: float = 1024,
    warmup: int = 3,
    iterations: int = 10,
    qps_per_connection: int = 4,
    split_data_on_qps: int = 1,
    channels_per_net_peer: int = 8,
    server_host: str = SERVER_HOST,
    client_host: str = CLIENT_HOST,
    master_port: int | None = None,
    show_network_log: bool = False,
) -> dict:
    """Measure one-way GPU-direct NCCL bandwidth on one RoCE rail."""
    if not 0 <= rail <= 7:
        raise ValueError("rail must be between 0 and 7")
    if qps_per_connection < 1 or channels_per_net_peer < 1:
        raise ValueError("QP and channel counts must be at least 1")
    if split_data_on_qps not in (0, 1):
        raise ValueError("split_data_on_qps must be 0 or 1")
    if payload_mb <= 0 or warmup < 0 or iterations < 1:
        raise ValueError("Invalid payload, warmup, or iteration count")
    if master_port is None:
        master_port = random.randint(29000, 29999)

    upload = subprocess.run(
        ["scp", str(LOCAL_NCCL_BENCHMARK), f"{server_host}:{REMOTE_NCCL_BENCHMARK}"],
        text=True, capture_output=True, timeout=30,
    )
    if upload.returncode != 0:
        raise RuntimeError(f"Could not upload benchmark:\n{upload.stderr}")

    common = dict(
        master_port=master_port, rail=rail, payload_mb=payload_mb,
        warmup=warmup, iterations=iterations,
        qps_per_connection=qps_per_connection,
        split_data_on_qps=split_data_on_qps,
        channels_per_net_peer=channels_per_net_peer,
    )
    leader_command = _nccl_command(node_rank=0, **common)
    worker_command = _nccl_command(node_rank=1, **common)

    print(
        f"GPU {rail} <-> mlx5_bond_{rail}: payload={payload_mb} MiB, "
        f"QPs={qps_per_connection}, split={split_data_on_qps}, "
        f"channels={channels_per_net_peer}"
    )
    worker = subprocess.Popen(
        ["ssh", client_host, worker_command],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    leader = None
    try:
        time.sleep(2)
        leader = subprocess.Popen(
            ["ssh", server_host, leader_command],
            text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        )
        leader_output, _ = leader.communicate(timeout=130)
        worker_output, _ = worker.communicate(timeout=15)
    finally:
        _stop_process(worker)
        if leader is not None:
            _stop_process(leader)

    if leader.returncode != 0 or worker.returncode != 0:
        raise RuntimeError(
            f"NCCL test failed.\nLEADER:\n{leader_output}\nWORKER:\n{worker_output}"
        )
    combined_output = leader_output + worker_output
    gpu_direct = "GDRDMA" in combined_output and "GPU Direct RDMA Enabled" in combined_output
    if not gpu_direct:
        raise RuntimeError(f"NCCL did not confirm GDRDMA:\n{combined_output}")

    result = _parse_nccl_result(leader_output)
    result.update(
        rail=rail, qps_per_connection=qps_per_connection,
        split_data_on_qps=split_data_on_qps,
        channels_per_net_peer=channels_per_net_peer,
        gpu_direct_confirmed=gpu_direct,
    )
    print(
        f"Result: {result['median_GBps']:.2f} GB/s median, "
        f"{result['median_ms']:.3f} ms; GDRDMA confirmed"
    )
    if show_network_log:
        for line in combined_output.splitlines():
            if any(token in line for token in ("GDRDMA", "GPU Direct", "NET/IB", "p2p channels")):
                print(line)
    return result


## Run one NCCL configuration

Change the three highlighted values and rerun this cell. A 1 GiB payload is large enough to measure bandwidth rather than startup latency.


In [9]:
nccl_result = run_nccl_roce_bw(
    rail=0,
    payload_mb=1024,
    qps_per_connection=4,      # NCCL_IB_QPS_PER_CONNECTION
    split_data_on_qps=1,       # NCCL_IB_SPLIT_DATA_ON_QPS
    channels_per_net_peer=8,   # NCCL_NCHANNELS_PER_NET_PEER
)
nccl_result

GPU 0 <-> mlx5_bond_0: payload=1024 MiB, QPs=8, split=1, channels=8
Result: 54.59 GB/s median, 19.668 ms; GDRDMA confirmed


{'iterations': 10,
 'max_ms': 22.001905366778374,
 'median_GBps': 54.59405724332036,
 'median_ms': 19.66774184256792,
 'min_latency_GBps': 56.24559095074385,
 'min_ms': 19.090239889919758,
 'p95_ms': 22.001905366778374,
 'payload_GB': 1.073741824,
 'payload_bytes': 1073741824,
 'rail': 0,
 'qps_per_connection': 8,
 'split_data_on_qps': 1,
 'channels_per_net_peer': 8,
 'gpu_direct_confirmed': True}

In [10]:
nccl_result = run_nccl_roce_bw(
    rail=0,
    payload_mb=1024,
    qps_per_connection=16,      # NCCL_IB_QPS_PER_CONNECTION
    split_data_on_qps=1,       # NCCL_IB_SPLIT_DATA_ON_QPS
    channels_per_net_peer=16,   # NCCL_NCHANNELS_PER_NET_PEER
)
nccl_result

GPU 0 <-> mlx5_bond_0: payload=1024 MiB, QPs=16, split=1, channels=16
Result: 28.97 GB/s median, 37.060 ms; GDRDMA confirmed


{'iterations': 10,
 'max_ms': 38.231759797781706,
 'median_GBps': 28.97317115563006,
 'median_ms': 37.059865426272154,
 'min_latency_GBps': 28.99194504942846,
 'min_ms': 37.03586710616946,
 'p95_ms': 38.231759797781706,
 'payload_GB': 1.073741824,
 'payload_bytes': 1073741824,
 'rail': 0,
 'qps_per_connection': 16,
 'split_data_on_qps': 1,
 'channels_per_net_peer': 16,
 'gpu_direct_confirmed': True}